# Agent Loops Study — Colab CPU Server Pipeline

This notebook runs the complete **Agent Loops Study (Phases 1 & 2)** on a **Google Colab CPU runtime** so that parsing 60,000+ trajectories, replaying loop detectors across 200 matched-random seeds, and running 1,000-replicate task-clustered bootstraps execute on remote CPUs without blocking your Mac.


## 1. Environment Check & Colab CPU Setup

Verify the active CPU server environment, install any missing packages, unpack the repository bundle on the remote Colab VM, and run the unit test suite.

In [ ]:
# @title 1. Setup Colab CPU Environment & Run Unit Tests
import importlib.util
import os
from pathlib import Path
import platform
import subprocess
import sys
import tarfile

print(f"Platform: {platform.platform()} | Python: {sys.version.split()[0]} | CPU cores: {os.cpu_count()}")

REQUIRED_PACKAGES = [
    "huggingface_hub",
    "pyarrow",
    "pandas",
    "numpy",
    "matplotlib",
    "orjson",
    "pytest",
    "requests",
    "tqdm",
    "scipy",
    "statsmodels",
]
missing = [pkg for pkg in REQUIRED_PACKAGES if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

ROOT = Path.cwd()
bundle_path = Path("/content/code_bundle.tar.gz")
if not (ROOT / "src" / "agent_loops").exists() and bundle_path.exists():
    print(f"Extracting {bundle_path} into /content ...")
    with tarfile.open(bundle_path, "r:gz") as tf:
        tf.extractall("/content")
    ROOT = Path("/content")

if (ROOT / "src" / "agent_loops").exists():
    SRC_DIR = ROOT / "src"
elif Path("/Users/davidflorez/.gemini/jetski/scratch/agent-loops/src/agent_loops").exists():
    ROOT = Path("/Users/davidflorez/.gemini/jetski/scratch/agent-loops")
    SRC_DIR = ROOT / "src"
else:
    raise FileNotFoundError(f"Could not find src/agent_loops in {ROOT}.")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

import pytest
ret = pytest.main(["-q", str(ROOT / "tests")])
assert ret == 0, f"Pytest failed with exit code {ret}"


## 2. Run Full Study Pipeline (Phases 1 & 2) and Display Visualizations

Executes `run_all.main()` (fixed seed `42`) on the Colab CPU VM and renders the summary table and all 6 publication plots inline.

In [ ]:
# @title 2. Execute Study Pipeline & Render Figures
from IPython.display import Image, Markdown, display
import pandas as pd
import run_all

run_all.main()

summary_df = pd.read_csv(ROOT / "results" / "open_vs_closed_summary.csv")
display(summary_df)

for png_name in [
    "step_vs_dollar_share_per_model.png",
    "detector_savings_vs_false_stop.png",
    "loop_prevalence_by_harness.png",
    "loops_per_100_by_model_band.png",
    "open_closed_gap_before_after_matching.png",
    "open_vs_closed_across_four_datasets.png",
]:
    img_path = ROOT / "assets" / png_name
    if img_path.exists():
        display(Markdown(f"### `{png_name}`"))
        display(Image(filename=str(img_path)))


### Q&A
- **How often do AI coding and terminal agents get stuck in pathological loops, and what share of spend do loops consume?** On Terminal-Bench 2.0 (34,462 step-logged trajectories), 31.00% (10,684) of runs contain at least one pathological loop, representing 7.38% of steps (79,344 / 1,075,547) and 17.71% ($589.76 / $3,330.95) of cache-aware spend across 5,011 cost-validated runs. On SWE-rebench July 2026 (9,435 trajectories), 54.99% (5,188) of runs exhibit a loop (4.11% of steps; 2.87% of pooled spend across 6,844 cost-validated runs).
- **Do open-weight models loop more than closed models once compared fairly within identical scaffolds, shared tasks, and capability bands?** Across all 7 uniform-scaffold cohorts, capability-band matching and task-clustered regressions substantially attenuate raw Open-vs-Closed gaps. Pooled within-scaffold Negative Binomial regression (with `log(steps)` offset and two-way `(model_id, task_id)` clustering) yields an adjusted Incidence Rate Ratio (IRR) of 1.31 (95% CI [0.81, 2.13], p = 0.2765), while binary any-loop Logit yields OR = 2.25 (95% CI [1.11, 4.53], p = 0.0236).

### Data Analysis Key Findings
- **Disproportionate Late-Trajectory Cost:** On Terminal-Bench 2.0, loop steps consume 17.71% of cache-aware dollars despite accounting for only 7.38% of steps.
- **Pass-Rate Penalty:** Trajectories containing pathological loops pass at significantly lower rates than loop-free runs across Terminal-Bench 2.0 (27.42% vs 34.27%), SWE-rebench (43.14% vs 59.17%), and τ²-bench (48.36% vs 72.45%).
- **Rule-Based Detectors Miss Multi-Step Cycles:** Standard 5-step exact-repeat detectors (`cline_style`) miss 82.93% (277/334) of `period_k_rotation (k <= 5)` runs on Terminal-Bench 2.0 and 98.33% (118/120) on SWE-rebench.

### Insights or Next Steps
- **Upgrade Harness Loop Detectors:** Combine canonicalized period-k (k <= 5) cycle and A-B-A thrash detection with adaptive `1.25x P90` harness turn budgets.
- **Remote CPU Execution:** Use `colab exec -s <session> --timeout 1800 -f agent_loops_colab_cpu.ipynb` to run future refreshes entirely on Google Colab CPUs.